In [1]:
import pandas as pd

# Load the master dataset we built in notebook 02
df = pd.read_csv("../data/processed/master_climate_risk.csv")

print(df.shape)
print(df.head())

(1150, 6)
   year state  price       sales   avg_temp  disaster_count
0  2001    AK  10.54    5454.080  66.458333             1.0
1  2001    AL   5.60   79358.258  63.033333             1.0
2  2001    AR   6.05   41732.449  61.225000             1.0
3  2001    AZ   7.27   62274.304  61.200000             1.0
4  2001    CA  11.22  247758.778  59.125000             0.0


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the scaler
scaler = MinMaxScaler()

#Formula for normalization: (x - min) / (max - min)
# Normalize all four risk variables
df['price_scaled'] = scaler.fit_transform(df[['price']])
df['sales_scaled'] = scaler.fit_transform(df[['sales']])
df['temp_scaled'] = scaler.fit_transform(df[['avg_temp']])
df['disaster_scaled'] = scaler.fit_transform(df[['disaster_count']])

print(df[['price_scaled', 'sales_scaled', 'temp_scaled', 'disaster_scaled']].head())

   price_scaled  sales_scaled  temp_scaled  disaster_scaled
0      0.177565      0.000252     0.859327             0.25
1      0.038331      0.151853     0.789501             0.25
2      0.051015      0.074670     0.752633             0.25
3      0.085400      0.116808     0.752124             0.25
4      0.196731      0.497298     0.709820             0.00


In [3]:
# Calculate composite risk score using weighted average
df['risk_score'] = (
    df['disaster_scaled'] * 0.40 +
    df['temp_scaled'] * 0.30 +
    df['price_scaled'] * 0.20 +
    df['sales_scaled'] * 0.10
)

print(df[['state', 'year', 'risk_score']].head(10))
print(f"\nRisk score range: {df['risk_score'].min():.3f} to {df['risk_score'].max():.3f}")

  state  year  risk_score
0    AK  2001    0.393336
1    AL  2001    0.359702
2    AR  2001    0.343460
3    AZ  2001    0.354398
4    CA  2001    0.302022
5    CO  2001    0.255884
6    CT  2001    0.194112
7    DE  2001    0.210065
8    FL  2001    0.544590
9    GA  2001    0.377594

Risk score range: 0.088 to 0.712


In [4]:
# Average risk score per state across all years
df_state_risk = df.groupby('state')['risk_score'].mean().sort_values(ascending=False).reset_index()
df_state_risk.columns = ['state', 'avg_risk_score']

print("Top 15 highest risk states:")
print(df_state_risk.head(15))

Top 15 highest risk states:
   state  avg_risk_score
0     FL        0.512944
1     TX        0.497956
2     CA        0.469153
3     AK        0.459695
4     OK        0.417297
5     AL        0.414153
6     AZ        0.410058
7     LA        0.404899
8     MS        0.396556
9     GA        0.381450
10    NC        0.367775
11    TN        0.362606
12    NM        0.353715
13    KY        0.351706
14    AR        0.348908


In [5]:
# Save state risk scores
df_state_risk.to_csv("../data/processed/state_risk_scores.csv", index=False)

# Save full master with risk scores
df.to_csv("../data/processed/master_with_risk_scores.csv", index=False)

print("Both files saved!")

Both files saved!
